# Who wants to be a PoliMillionaire? — Strategy Pattern

**Coding assistants used:** Claude Code (Anthropic)

---

This notebook implements the **Strategy Pattern** for solving quiz questions with a local LLM.
Each strategy (zero-shot, few-shot, math) is injected into `QuizSolver` at construction time —
no internal branching, no classifier. The solver is a callable compatible with both
`client.play_game(comp.id, solver)` and the manual game loop in `analysis.ipynb`.

In [2]:
from __future__ import annotations
import json
import math
import os
import re
import sys
import time
from abc import ABC, abstractmethod
from datetime import datetime
from pathlib import Path
from typing import Optional

import torch
from transformers import BitsAndBytesConfig, pipeline
import dotenv

c:\Users\andre\Desktop\ANDREA\POLIMI\Natural Language Processing\NLP_project_repo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
BASE_DIR = Path('./')
RUNS_DIR = BASE_DIR / 'data' / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

package_parent_dir = os.path.abspath('./')
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

from millionaire_client import MillionaireClient, AuthenticationError
dotenv.load_dotenv()

username = os.getenv('USER')
pwd      = os.getenv('PASSWORD')
API_URL  = 'http://131.175.15.22:51111/'

client = MillionaireClient(API_URL)
try:
    user = client.login(username, pwd)
    print('Login successful. Welcome,', user.username)
except AuthenticationError as e:
    print('Login failed:', e)

MillionaireError: Could not connect to server at http://131.175.15.22:51111: ('Connection aborted.', ConnectionResetError(10054, "Connessione in corso interrotta forzatamente dall'host remoto", None, 10054, None))

In [ ]:
MODEL_NAME       = 'Qwen/Qwen2.5-1.5B-Instruct'  # public model, no auth needed
USE_QUANTIZATION = True                            # 4-bit quantization
REASONING        = False                           # True to enable thinking mode
NUM_RUNS         = 10                             # game repetitions per competition
RUN_COMMENT      = 'strategy-pattern few-shot'   # free-text description of this experiment
TECHNIQUES       = ['few-shot', 'greedy', 'strategy-pattern']

MAX_LEVELS = 15  # max levels for all competitions

In [ ]:
def _sanitize(text, max_len=24):
    # Safe for filesystem, the name must be
    return ''.join(c if c.isalnum() or c in '_-' else '_' for c in text)[:max_len].strip('_')


def create_run_folder(model_name):
    ts = datetime.now().strftime('%d%m%Y%H%M')
    name = _sanitize(model_name) + '_' + ts
    run_dir = RUNS_DIR / name
    run_dir.mkdir(parents=True, exist_ok=True)
    print('Run folder created:', run_dir.name)
    return run_dir


def save_metadata(run_dir, model_name, reasoning, comment, techniques, num_runs):
    # Write run_info.json early so metadata survives a mid-run crash
    meta = {
        'model_name': model_name,
        'reasoning': reasoning,
        'comment': comment,
        'techniques': techniques,
        'timestamp': datetime.now().isoformat(),
        'num_runs_per_competition': num_runs,
    }
    with open(run_dir / '.run_info.json', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2)
    return meta


def compute_competition_summary(attempts_log):
    # Accuracy = correct / (num_attempts * max_levels)
    # Penalizes early exits: finishing at level 5 of 15 contributes 4/15, not 4/4
    total_q = total_l = correct_q = timeout_q = 0
    total_time = 0.0
    all_earned = []
    max_level_reached = 0
    for attempt in attempts_log:
        max_level_reached = attempt['final_level'] if attempt['final_level'] > max_level_reached else max_level_reached
        total_l += attempt['final_level']
        all_earned.append(attempt['earned'])
        for q in attempt['questions']:
            total_q += 1
            total_time += q['time']
            if q['timed_out']:
                timeout_q += 1
            elif q['correct'] is True:
                correct_q += 1
    denominator = len(attempts_log) * MAX_LEVELS
    return {
        'total_attempts': len(attempts_log),
        'questions_answered': total_q,
        'avg_level': round(total_l / len(attempts_log), 2) if attempts_log else 0,
        'max_level': max_level_reached,
        'avg_earned': round(sum(all_earned) / len(all_earned), 2) if all_earned else 0,
        'max_earned': max(all_earned) if all_earned else 0,
        'accuracy': round(correct_q / denominator, 4) if denominator > 0 else 0,
        'timeout_count': timeout_q,
        'avg_time_per_question': round(total_time / total_q, 4) if total_q > 0 else 0,
        'max_time': round(max(q['time'] for attempt in attempts_log for q in attempt['questions']), 4) if total_q > 0 else 0,
    }


def save_competition_result(run_dir, comp, attempts_log):
    filename = 'comp_' + str(comp.id) + '_' + _sanitize(comp.name) + '.json'
    data = {
        'competition_id': comp.id,
        'competition_name': comp.name,
        'summary': compute_competition_summary(attempts_log),
        'attempts': attempts_log,
    }
    with open(run_dir / filename, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print('  Saved:', filename)


def save_run_summary(run_dir, all_comp_data, metadata):
    total_attempts = total_questions = correct_total = 0
    denominator_total = 0
    per_comp = {}
    for comp_id, comp_info in all_comp_data.items():
        s = compute_competition_summary(comp_info['attempts'])
        per_comp[str(comp_id)] = {
            'name': comp_info['name'],
            'accuracy': s['accuracy'],
            'avg_level': s['avg_level'],
            'avg_earned': s['avg_earned'],
        }
        total_attempts += s['total_attempts']
        total_questions += s['questions_answered']
        denominator_total += s['total_attempts'] * comp_info['max_levels']
        for attempt in comp_info['attempts']:
            for q in attempt['questions']:
                if q['correct'] is True:
                    correct_total += 1
    run_info = {
        **metadata,
        'total_competitions': len(all_comp_data),
        'overall_accuracy': round(correct_total / denominator_total, 4) if denominator_total > 0 else 0,
        'comp_info': per_comp,
    }
    with open(run_dir / '.run_info.json', 'w', encoding='utf-8') as f:
        json.dump(run_info, f, indent=2)
    print('  Saved: .run_info.json')
    return run_info


def update_runs_log(runs_dir, run_folder, run_info):
    log_path = runs_dir / 'runs_log.json'
    entries = []
    if log_path.exists():
        with open(log_path, 'r', encoding='utf-8') as f:
            entries = json.load(f)
    competition_summary = {
        info['name']: info['accuracy']
        for info in run_info['comp_info'].values()
    }
    entries.append({
        'run_folder':           run_folder,
        'model_name':           run_info['model_name'],
        'total_runs':           run_info['num_runs_per_competition'],
        'overall_accuracy':     run_info['overall_accuracy'],
        'reasoning':            run_info['reasoning'],
        'comment':              run_info['comment'],
        'techniques':           run_info['techniques'],
        'competition_accuracy': competition_summary,
    })
    with open(log_path, 'w', encoding='utf-8') as f:
        json.dump(entries, f, indent=2)
    print('runs_log.json updated -', len(entries), 'runs logged')

In [ ]:
class QuizStrategy(ABC):
    """
    Contratto comune per tutte le strategie di risposta.
    Riceve un oggetto Question del client, restituisce option.id.
    Non conosce il modello: ci pensa QuizSolver.
    """
    _LETTERS = ['A', 'B', 'C', 'D']
    max_new_tokens: int = 20  # default: just a letter, fast it must be

    @abstractmethod
    def build_messages(self, question) -> list[dict]:
        """Builds chat messages [{'role': ..., 'content': ...}] from a Question object."""
        pass

    def parse_answer(self, raw: str, question) -> int:
        """
        Extracts letter from raw LLM output, maps it to option.id.
        Subclasses may override for domain-specific parsing.
        """
        opts = question.options[:4]

        m = re.search(r'Answer\s*[:\-]?\s*([A-Da-d])', raw)
        if m:
            letter = m.group(1).upper()
        else:
            m = re.search(r'\b([A-Da-d])\b', raw)
            letter = m.group(1).upper() if m else None

        if letter and letter in self._LETTERS:
            idx = self._LETTERS.index(letter)
            if idx < len(opts):
                return opts[idx].id

        # Fallback  first option
        print('WARNING: no letter parsed from:', repr(raw[:60]))
        return opts[0].id


# ── Few-shot examples for general knowledge ──────────────────────────────────

_FEW_SHOT_EXAMPLES = """\
Example 1:
Question: What is the capital of France?
Options:
A: Rome
B: Dublin
C: Paris
D: Oslo
Answer with just the letter of the correct option (A, B, C, or D).
Answer: C

Example 2:
Question: In which year did the French Revolution begin?
Options: 
A: 1776  
B: 1799  
C: 1789  
D: 1804
Answer with just the letter of the correct option (A, B, C, or D).
Answer: C

"""

# ── Few-shot examples for math: show when to use code vs direct answer ────────

_MATH_FEW_SHOT_EXAMPLES = """\
Example 1 — computation needed:
Question: What is 12 multiplied by 7?
Options:
A: 72
B: 74
C: 84
D: 96
```python
12 * 7
```

Example 2 — no computation needed:
Question: how many sides does a triangle have?
Options:
A: 9
B: 4
C: 1
D: 3
Answer: D

"""


class KnowledgeStrategy(QuizStrategy):
    """
    Zero-shot or few-shot strategy for general knowledge questions.
    Optional RAG context can be injected at construction time.
    """
    max_new_tokens: int = 20  # just "Answer: X", nothing more needed

    def __init__(self, few_shot: bool = False, rag_context: Optional[str] = None):
        self.few_shot    = few_shot
        self.rag_context = rag_context

    def build_messages(self, question) -> list[dict]:
        opts = question.options[:4]
        choices_str = '\n'.join(
            f'{self._LETTERS[i]}: {opt.text}' for i, opt in enumerate(opts)
        )
        system = (
            'You are a knowledgeable quiz assistant. '
            'Read the question and all choices carefully, then pick the best answer. '
            'End your reply with exactly: Answer: <letter>'
        )
        parts: list[str] = []
        if self.few_shot:
            parts.append(_FEW_SHOT_EXAMPLES)
        if self.rag_context:
            system += '\nUse the provided reference context, written between <context> tags, to inform your answer'
            parts.append(f'Reference context (use only if relevant):\n<context>{self.rag_context}</context>\n')
        parts.append(f'Question: {question.text}\nChoices:\n{choices_str}')
        parts.append('\nThink briefly, then write Answer: <A/B/C/D>')
        return [
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': '\n'.join(parts)},
        ]


def _safe_eval(expr: str) -> Optional[float]:
    """Evaluates math expressions safely — exec outside this sandbox, not allowed it is."""
    allowed = {
        'sqrt': math.sqrt, 'log': math.log, 'log10': math.log10,
        'sin': math.sin,   'cos': math.cos,  'tan': math.tan,
        'pi': math.pi,     'e': math.e,
        'abs': abs,        'round': round,   'pow': pow,
    }
    sanitized = re.sub(r'[^0-9\+\-\*\/\(\)\.\s\_a-zA-Z]', '', expr)
    try:
        return eval(sanitized, {'__builtins__': {}}, allowed)  # noqa: S307
    except Exception:
        return None


class MathStrategy(QuizStrategy):
    """
    Strategy for math/numerical questions.
    Few-shot examples teach the model when to emit a ```python``` block
    vs when to reply directly with Answer: <letter>.

    Kept short on purpose: 60 tokens ≈ 12s at 5 tok/s, well within the 30s timeout.
    """
    max_new_tokens: int = 60  # one code block or a letter — no essays, please

    def __init__(self, few_shot: bool = True):
        self.few_shot = few_shot

    def build_messages(self, question) -> list[dict]:
        opts = question.options[:4]
        choices_str = '\n'.join(
            f'{self._LETTERS[i]}: {opt.text}' for i, opt in enumerate(opts)
        )
        system = (
            'You are a math solver. '
            'If a computation is needed, write ONE Python expression in a ```python``` block. '
            'Otherwise reply with exactly: Answer: <letter>. '
            'Be concise — one line of output only.'
        )
        parts: list[str] = []
        if self.few_shot:
            parts.append(_MATH_FEW_SHOT_EXAMPLES)
        parts.append(f'Question: {question.text}\nOptions:\n{choices_str}')
        return [
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': '\n'.join(parts)},
        ]

    def parse_answer(self, raw: str, question) -> int:
        opts = question.options[:4]
        code_block = re.search(r'```(?:python)?\s*([\s\S]+?)```', raw)
        if code_block:
            expr = code_block.group(1).strip()
            print(f'[MathStrategy] code block found: {repr(expr)}')
            result = _safe_eval(expr)
            print(f'[MathStrategy] eval result: {result}')
            if result is not None:
                for opt in opts:
                    try:
                        if abs(float(opt.text) - result) < 1e-6:
                            print(f'[MathStrategy] numeric match → option "{opt.text}"')
                            return opt.id
                    except (ValueError, TypeError):
                        pass
                print('[MathStrategy] no numeric match, falling back to letter parsing')
        else:
            print('[MathStrategy] no code block found, falling back to letter parsing')
        return super().parse_answer(raw, question)

In [ ]:
print('Loading model:', MODEL_NAME)
print('Quantization:', USE_QUANTIZATION)
print('This may take a few minutes on first run...')

# 4-bit quantization config — memory halved, quality preserved almost entirely
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',           # NormalFloat4: optimal for LLM weights it is
    bnb_4bit_use_double_quant=True,      # Extra compression of scale factors, yes
    bnb_4bit_compute_dtype=torch.bfloat16,
) if USE_QUANTIZATION else None

llm_pipeline = pipeline(
    'text-generation',
    model=MODEL_NAME,
    model_kwargs={'quantization_config': bnb_config} if bnb_config else {},
    dtype=torch.bfloat16 if not USE_QUANTIZATION else None,
    device_map='auto',
    trust_remote_code=False,
)

if torch.cuda.is_available():
    mem_gb = torch.cuda.memory_allocated() / 1024**3
    print(f'GPU memory used: {mem_gb:.1f} GB')

print('Model loaded!')

Loading model: Qwen/Qwen2.5-1.5B-Instruct
Quantization: True
This may take a few minutes on first run...


Loading weights:  15%|█▍        | 50/338 [00:07<00:40,  7.14it/s]


KeyboardInterrupt: 

In [ ]:
def _generate(messages: list[dict], max_new_tokens: int) -> str:
    """Formats messages with chat template and generates text"""
    try:
        prompt_text = llm_pipeline.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        prompt_text = messages[-1]['content']

    n_prompt_tok = len(llm_pipeline.tokenizer.encode(prompt_text))
    print('[PROMPT] ' + '-' * 50)
    print(prompt_text)
    print(f'[tokens: prompt={n_prompt_tok}, max_new={max_new_tokens}]')
    print('-' * 60)

    outputs = llm_pipeline(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    raw = outputs[0]['generated_text'].strip()

    n_out_tok = len(llm_pipeline.tokenizer.encode(raw))
    print(f'[RAW OUTPUT — {n_out_tok} tokens] ' + '-' * 30)
    print(raw)
    print('-' * 60)

    return raw


def _letter_from_id(option_id: int, question) -> str:
    """Maps option.id back to A/B/C/D for logging"""
    letters = ['A', 'B', 'C', 'D']
    for i, opt in enumerate(question.options[:4]):
        if opt.id == option_id:
            return letters[i] if i < len(letters) else '?'
    return '?'


class QuizSolver:
    """
    Wraps a QuizStrategy into a callable compatible with client.play_game() and
    the manual game loop. Uses the global llm_pipeline — loaded once, reused always.
    Token budget is read from strategy.max_new_tokens (overridable via constructor).

    Usage:
        solver = QuizSolver(KnowledgeStrategy(few_shot=True))
        session = client.play_game(comp.id, solver)
    """

    def __init__(self, strategy: QuizStrategy, max_new_tokens: Optional[int] = None):
        self.strategy       = strategy
        # Use strategy's own budget unless explicitly overridden
        self.max_new_tokens = max_new_tokens if max_new_tokens is not None else strategy.max_new_tokens
        self.last_raw_output    = None
        self.last_chosen_letter = None
        self.last_chosen_text   = None
        self.max_time           = 0.0

    def __call__(self, question) -> int:
        if not question or not question.options:
            raise ValueError('No question or options provided')

        strategy_name = type(self.strategy).__name__
        print(f'\n[{strategy_name}] Question: {question.text}')
        opts = question.options[:4]
        for i, opt in enumerate(['A', 'B', 'C', 'D'][:len(opts)]):
            print(f'  {opt}) {opts[i].text}')

        messages = self.strategy.build_messages(question)

        t0      = time.time()
        raw     = _generate(messages, self.max_new_tokens)
        elapsed = time.time() - t0

        self.max_time = max(self.max_time, elapsed)
        option_id     = self.strategy.parse_answer(raw, question)

        self.last_raw_output    = raw
        self.last_chosen_letter = _letter_from_id(option_id, question)
        chosen_opt              = question.get_option_by_id(option_id)
        self.last_chosen_text   = chosen_opt.text if chosen_opt else '?'

        print(f'[{strategy_name}] → chose {self.last_chosen_letter}) {self.last_chosen_text}  ({elapsed:.2f}s)')

        return option_id

In [ ]:
# Create run folder and save metadata
run_dir  = create_run_folder(MODEL_NAME)
metadata = save_metadata(run_dir, MODEL_NAME, REASONING, RUN_COMMENT, TECHNIQUES, NUM_RUNS)

comps = client.competitions.list_all()
print('Competitions found:', len(comps))
all_comp_data = {}

for comp in comps:
    if comp.id!=3:
        continue  # Skip non-math competitions for this run
    print()
    print('=' * 60)
    print('Competition', comp.id, ':', comp.name)
    attempts_log = []
    
    strategy = MathStrategy() if comp.id==3 else KnowledgeStrategy(few_shot='few-shot' in TECHNIQUES)

    for run_idx in range(1, NUM_RUNS + 1):
        print('  Run', run_idx, '/', NUM_RUNS, '...', end=' ', flush=True)
        session = client.game.start(comp.id)
        solver  = QuizSolver(strategy)
        questions_log = []

        last_correct_level = 0
        total_earned = 0

        while session.in_progress:
            question = session.current_question
            if question is None:
                break

            letters      = ['A', 'B', 'C', 'D']
            opts         = question.options[:4]
            options_dict = {letters[i]: opts[i].text for i in range(len(opts))}

            current_level = session.state.current_level
            current_prize = next(
                (ml.amount for ml in session.state.money_pyramid if ml.level == current_level),
                None,
            )

            t0        = time.time()
            option_id = solver(question)
            result    = session.answer(option_id)
            elapsed   = time.time() - t0

            if result.correct:
                last_correct_level = current_level
                total_earned += current_prize or 0

            questions_log.append({
                'level':          current_level,
                'prize':          current_prize,
                'question_text':  question.text,
                'options':        options_dict,
                'llm_raw_output': solver.last_raw_output,
                'chosen_letter':  solver.last_chosen_letter,
                'correct':        result.correct,
                'timed_out':      result.timed_out,
                'time':           elapsed,
            })

        print(
            'earned=' + str(total_earned)
            + ', q=' + str(len(questions_log))
            + ', max_t=' + '{:.2f}'.format(solver.max_time) + 's'
        )
        attempts_log.append({
            'attempt':     run_idx,
            'earned':      total_earned,
            'final_level': last_correct_level,
            'questions':   questions_log,
        })

    save_competition_result(run_dir, comp, attempts_log)
    all_comp_data[comp.id] = {
        'name':       comp.name,
        'max_levels': comp.max_levels,
        'attempts':   attempts_log,
    }

run_info = save_run_summary(run_dir, all_comp_data, metadata)
update_runs_log(RUNS_DIR, run_dir.name, run_info)
print()
print('Run complete! Overall accuracy:', '{:.1%}'.format(run_info['overall_accuracy']))

Run folder created: Qwen_Qwen2_5-1_5B-Instru_170520262321
Competitions found: 4

Competition 0 : Entertainment
  Run 1 / 2 ... 

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[KnowledgeStrategy] Question: Which of the following best describes the significance of the Tramp character in Chaplin's films?
  A) The Tramp was primarily used as a vehicle for slapstick comedy with no deeper meaning.
  B) The Tramp was a parody of the wealthy and privileged, making fun of their lifestyle.
  C) The Tramp represented the 'little man' and his struggles, becoming a universal symbol of resilience and dignity.
  D) The Tramp was a one-dimensional character with no significant impact on Chaplin's career.
[PROMPT] --------------------------------------------------
<|im_start|>system
You are a knowledgeable quiz assistant. Read the question and all choices carefully, then pick the best answer. End your reply with exactly: Answer: <letter><|im_end|>
<|im_start|>user
Example 1:
Question: What is the capital of France?
Options:
A: Rome
B: Dublin
C: Paris
D: Oslo
Answer with just the letter of the correct option (A, B, C, or D).
Answer: C

Example 2:
Question: In which year did

[transformers] Both `max_new_tokens` (=20) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[RAW OUTPUT] ----------------------------------------------
Answer: C
------------------------------------------------------------
[KnowledgeStrategy] Generation time: 11.59 seconds
[KnowledgeStrategy] Raw output: 'Answer: C'...
[KnowledgeStrategy] → chose C) The Tramp represented the 'little man' and his struggles, becoming a universal symbol of resilience and dignity.  (11.59s)

[KnowledgeStrategy] Question: What term describes Judi Dench's involvement in the revival of the York Mystery Plays in 1951, 1954, and 1957?
  A) Academic research
  B) Non-professional theatrical involvement
  C) Film production
  D) Professional acting debut
[PROMPT] --------------------------------------------------
<|im_start|>system
You are a knowledgeable quiz assistant. Read the question and all choices carefully, then pick the best answer. End your reply with exactly: Answer: <letter><|im_end|>
<|im_start|>user
Example 1:
Question: What is the capital of France?
Options:
A: Rome
B: Dublin
C: Paris
D: 

...

[RAW OUTPUT] ----------------------------------------------
Answer: B
------------------------------------------------------------
[KnowledgeStrategy] Generation time: 8.54 seconds
[KnowledgeStrategy] Raw output: 'Answer: B'...
[KnowledgeStrategy] → chose B) converted from solar energy.  (8.54s)

[KnowledgeStrategy] Question: The roots of a plant can grow through cracks in a driveway, eventually causing the driveway to break. Which kind of process would be taking place?
  A) weathering
  B) sedimentation
  C) erosion
  D) exfoliation
[PROMPT] --------------------------------------------------
<|im_start|>system
You are a knowledgeable quiz assistant. Read the question and all choices carefully, then pick the best answer. End your reply with exactly: Answer: <letter><|im_end|>
<|im_start|>user
Example 1:
Question: What is the capital of France?
Options:
A: Rome
B: Dublin
C: Paris
D: Oslo
Answer with just the letter of the correct option (A, B, C, or D).
Answer: C

Example 2:
Question: I

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[MathStrategy] Question: Which among the following would result in the narrowest confidence interval?
  A) Large sample size and 95% confidence
  B) Small sample size and 99% confidence
  C) Large sample size and 99% confidence
  D) Small sample size and 95% confidence
[PROMPT] --------------------------------------------------
<|im_start|>system
You are a precise math solver. Reason step by step. If you need to compute something, write the expression inside a ```python``` block , this will be your answer.otherwise reply with exactly: Answer: <letter><|im_end|>
<|im_start|>user
Question: Which among the following would result in the narrowest confidence interval?
Options:
A: Large sample size and 95% confidence
B: Small sample size and 99% confidence
C: Large sample size and 99% confidence
D: Small sample size and 95% confidence

reason step by step, then answer as instructed<|im_end|>
<|im_start|>assistant

------------------------------------------------------------


KeyboardInterrupt: 

KeyboardInterrupt: 